<a href="https://colab.research.google.com/github/RajeevKambhatla/longevity-drug-repurposing/blob/main/19_final_candidates.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q shap anthropic pydantic tenacity
from google.colab import drive, userdata
drive.mount('/content/drive')

import json, numpy as np, pandas as pd
BASE = '/content/drive/MyDrive/longevity_project'
kd = json.load(open(f'{BASE}/data/processed/known_drugs.json'))
compounds = pd.read_csv(f'{BASE}/data/processed/compounds.csv', index_col='name')
sigpca = pd.read_parquet(f'{BASE}/data/processed/signature_pca.parquet')
fp = pd.read_parquet(f'{BASE}/data/processed/ecfp4.parquet')
T = pd.read_parquet(f'{BASE}/data/processed/targets.parquet')
final = pd.read_parquet(f'{BASE}/results/final_scores.parquet')
ranked = pd.read_csv(f'{BASE}/results/final_candidates.csv')
holdout = json.load(open(f'{BASE}/data/processed/holdout.json'))
background = json.load(open(f'{BASE}/data/processed/background_negatives.json'))
kind, feat = json.load(open(f'{BASE}/results/ml_decisions.json'))['setup'].split('|')
HALLMARKS = list(kd['by_hallmark'])

from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

def training_set(hm):
    k = kd['by_hallmark'][hm]
    pos = [d for d in k['helpful'] if d in compounds.index and d not in holdout[hm]]
    neg = [d for d in k['harmful'] if d in compounds.index] + background
    return pos, neg

if feat == 'all':
    has = compounds['has_structure'].values.astype(bool)
    fp_small = np.zeros((len(fp), 50), dtype='float32')
    fp_small[has] = TruncatedSVD(n_components=50, random_state=0).fit_transform(fp.values[has].astype('float32'))
    fpf = pd.DataFrame(fp_small, index=fp.index, columns=[f'fpsvd_{i}' for i in range(50)])
    fpf['has_structure'] = has.astype('float32')
    X = pd.concat([sigpca, fpf, T.astype('float32')], axis=1)
else:
    X = sigpca

def fit(pos, neg, seed=0):
    if kind == 'logistic':
        model = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, class_weight='balanced', max_iter=3000))
    else:
        model = RandomForestClassifier(n_estimators=300, min_samples_leaf=2, class_weight='balanced_subsample',
                                       n_jobs=-1, random_state=seed)
    y = np.r_[np.ones(len(pos)), np.zeros(len(neg))]
    return model.fit(X.loc[pos + neg].values, y)

print(len(ranked), 'candidate drug-hallmark pairs from Session 17 | ML setup:', kind, feat)

Mounted at /content/drive
90 candidate drug-hallmark pairs from Session 17 | ML setup: logistic signatures


In [2]:
from sklearn.metrics.pairwise import cosine_similarity

toxic = sorted({d for k in kd['by_hallmark'].values() for d in k['harmful'] if d in sigpca.index})
centroid = sigpca.loc[toxic].mean().values[None, :]
stress_pct = pd.Series(cosine_similarity(sigpca.values, centroid)[:, 0], index=sigpca.index).rank(pct=True) * 100
strength_pct = pd.Series(np.linalg.norm(sigpca.values, axis=1), index=sigpca.index).rank(pct=True) * 100

print('Toxic reference drugs:', ', '.join(toxic))
print(f"Their own toxic-likeness (median percentile): {stress_pct[toxic].median():.0f}")
print(f"Sirolimus's toxic-likeness: {stress_pct.get('sirolimus', np.nan):.0f}")

Toxic reference drugs: antimycin-a, bortezomib, brefeldin-a, camptothecin, chloroquine, doxorubicin, etoposide, geldanamycin, mg-132, mitomycin-c, palbociclib, puromycin, rotenone, tanespimycin, temozolomide, thapsigargin, tunicamycin
Their own toxic-likeness (median percentile): 98
Sirolimus's toxic-likeness: 71


In [3]:
import shap

def feature_group(c):
    if c.startswith('pc_'):
        return 'gene activity'
    if c.startswith('tg_') or c == 'has_targets':
        return 'targets'
    return 'chemistry'
col_group = np.array([feature_group(c) for c in X.columns])

top = ranked[~ranked['tier'].str.startswith('4')].head(20)
rows = []
for hm in top['hallmark'].unique():
    pos, neg = training_set(hm)
    model = fit(pos, neg)
    drugs = top.loc[top['hallmark'] == hm, 'drug'].tolist()
    Xd = X.loc[drugs].values
    if kind == 'forest':
        sv = shap.TreeExplainer(model).shap_values(Xd)
        sv = sv[1] if isinstance(sv, list) else (sv[:, :, 1] if sv.ndim == 3 else sv)
    else:
        scaler, lr = model.named_steps['standardscaler'], model.named_steps['logisticregression']
        sv = (scaler.transform(Xd) - scaler.transform(X.loc[pos + neg].values).mean(axis=0)) * lr.coef_[0]
    for d, row in zip(drugs, sv):
        total = np.abs(row).sum() or 1
        share = {g: np.abs(row[col_group == g]).sum() / total * 100 for g in ['gene activity', 'chemistry', 'targets']}
        helpful_targets = [X.columns[i][3:] for i in np.argsort(-row) if X.columns[i].startswith('tg_') and row[i] > 0][:3]
        rows.append({'drug': d, 'hallmark': hm, **{f'{g} %': round(v) for g, v in share.items()},
                     'targets pushing score up': ', '.join(helpful_targets) or '-'})

explain = pd.DataFrame(rows)
print(explain.to_string(index=False))

                drug         hallmark  gene activity %  chemistry %  targets % targets pushing score up
         kenpaullone     mitochondria              100            0          0                        -
orthothymotinic-acid     mitochondria              100            0          0                        -
           cl-218872     mitochondria              100            0          0                        -
     androstenedione     mitochondria              100            0          0                        -
       necrostatin-1     inflammation              100            0          0                        -
          ku-c103655     inflammation              100            0          0                        -
           as-605240       senescence              100            0          0                        -
            u-54494a       senescence              100            0          0                        -
          ku-c103655       senescence              100          

In [4]:
r = ranked.copy()
r['toxic_likeness'] = stress_pct.reindex(r['drug']).values.round(0)
r['signal_strength'] = strength_pct.reindex(r['drug']).values.round(0)
r['gene_list_pct'] = [final.loc[d, f'{h}_baseline'] for d, h in zip(r['drug'], r['hallmark'])]
r['ml_pct'] = [final.loc[d, f'{h}_ml'] for d, h in zip(r['drug'], r['hallmark'])]
r = r.merge(explain, on=['drug', 'hallmark'], how='left')

def red_flags(x):
    f = []
    if x.stress_concern: f.append('literature reports toxicity or cell stress')
    if x.toxic_likeness >= 95: f.append('signature resembles toxic drugs')
    if x.signal_strength >= 98: f.append('extremely strong signature')
    return '; '.join(f)

def cautions(x):
    f = []
    if pd.notna(x.ml_uncertainty) and x.ml_uncertainty >= 15: f.append('unstable ML score')
    if abs(x.gene_list_pct - x.ml_pct) >= 40: f.append('gene list and ML disagree')
    if pd.notna(x.get('chemistry %')) and x['chemistry %'] >= 50: f.append('score driven mainly by chemical similarity')
    return '; '.join(f)

r['red_flags'] = r.apply(red_flags, axis=1)
r['cautions'] = r.apply(cautions, axis=1)
r.to_csv(f'{BASE}/results/final_candidates_checked.csv', index=False)

print('Pairs with red flags:', (r['red_flags'] != '').sum(), 'of', len(r))
clean = r[(r['red_flags'] == '') & r['tier'].str[0].isin(['1', '2'])]
print(f'\nClean candidates with High or Medium evidence ({len(clean)}):')
print(clean[['drug', 'hallmark', 'tier', 'meta_pct', 'gene_list_pct', 'ml_pct', 'cautions']].round(0).to_string(index=False))

Pairs with red flags: 76 of 90

Clean candidates with High or Medium evidence (3):
         drug     hallmark              tier  meta_pct  gene_list_pct  ml_pct                                     cautions
  kenpaullone mitochondria   1 High evidence     100.0          100.0    92.0                                             
necrostatin-1 inflammation 2 Medium evidence      99.0          100.0    69.0                            unstable ML score
    as-605240   senescence 2 Medium evidence      91.0          100.0    30.0 unstable ML score; gene list and ML disagree


In [5]:
import anthropic
from pydantic import BaseModel
from tenacity import retry, stop_after_attempt, wait_exponential
client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))
MODEL = json.load(open(f'{BASE}/data/processed/review_prompt.json'))['model']

usable = r[r['red_flags'] == '']
blocks = []
for d, g in usable.groupby('drug'):
    lines = [f"  - {x.hallmark}: final score {x.meta_pct:.0f}th percentile; evidence {x.confidence}; "
             f"ML uncertainty {'±%.0f' % x.ml_uncertainty if pd.notna(x.ml_uncertainty) else 'n/a'}; "
             f"cautions: {x.cautions or 'none'}; mechanism: {str(x.mechanism)[:250]}; safety: {str(x.safety)[:200]}"
             for x in g.itertuples()]
    blocks.append(f"{d}:\n" + '\n'.join(lines))

class BestPer(BaseModel):
    hallmark: str
    drug: str
    why: str

class Pick(BaseModel):
    rank: int
    drug: str
    hallmarks: list[str]
    why: str
    main_caveat: str

class FinalPicks(BaseModel):
    best_per_hallmark: list[BestPer]
    top5: list[Pick]

SYSTEM = """You are a skeptical aging-biology reviewer making final selections from a drug-repurposing screen.
Use ONLY the candidate information provided. Weigh, in order: strength of published evidence (High over Medium over Low),
benefit on more than one hallmark, stable ML scores (small uncertainty), and absence of cautions. Screening scores alone
are weak evidence. For each of the six hallmarks, name the single best candidate, or "none" if no candidate is
reasonably supported. Then rank up to five candidates that look most promising across multiple hallmarks; choose fewer
than five if fewer are justified. Every drug you name must appear in the candidate list. For each pick, state the
single most important caveat. Respond with only a JSON object with keys best_per_hallmark (a list of objects with keys
hallmark, drug, why) and top5 (a list of objects with keys rank, drug, hallmarks, why, main_caveat)."""

@retry(stop=stop_after_attempt(3), wait=wait_exponential(min=2, max=20))
def pick():
    msg = client.messages.create(model=MODEL, max_tokens=8000, system=SYSTEM, messages=[{'role': 'user',
          'content': 'Hallmarks: ' + ', '.join(HALLMARKS) + '\n\nCandidates:\n' + '\n\n'.join(blocks)}])
    raw = next(b.text for b in msg.content if b.type == 'text')
    return FinalPicks.model_validate_json(raw[raw.find('{'): raw.rfind('}') + 1])

picks = pick()
allowed = set(usable['drug']) | {'none'}
named_drugs = [b.drug for b in picks.best_per_hallmark] + [p.drug for p in picks.top5]
print('Names not in the candidate list:', [d for d in named_drugs if d not in allowed] or 'none')

print('\nBEST PER HALLMARK')
for b in picks.best_per_hallmark:
    print(f'  {b.hallmark:17} {b.drug:22} {b.why}')
print('\nTOP 5 MULTI-HALLMARK CANDIDATES')
for p in picks.top5:
    print(f'  {p.rank}. {p.drug} ({", ".join(p.hallmarks)})\n     Why: {p.why}\n     Caveat: {p.main_caveat}')

Names not in the candidate list: none

BEST PER HALLMARK
  senescence        as-605240              Medium-strength evidence for PI3Kγ-mediated suppression of inflammatory/immune signaling that could indirectly limit senescence-associated inflammation; better evidence tier than the Low-evidence alternatives (cay-10415, ku-c103655, u-54494a).
  nutrient_sensing  tgx-115                Only candidate flagged for this hallmark; notably stable ML score (±0 uncertainty) and no cautions, though evidence tier is Low and mechanism is unreported.
  mitochondria      kenpaullone            High-strength published evidence with a clear, multi-model mechanism (GSK-3α/β and CDK2 inhibition preserving mitochondrial membrane integrity, reducing ROS/apoptosis), low ML uncertainty (±12), and no cautions flagged.
  autophagy         emf-bca1-16            Only candidate associated with this hallmark, but support is weak (Low evidence, unstable ML score, gene list/ML disagreement, no reported mechanism).

In [6]:
results = {'best_per_hallmark': [b.model_dump() for b in picks.best_per_hallmark],
           'top5': [p.model_dump() for p in picks.top5],
           'red_flag_rules': ['literature reports toxicity or cell stress', 'toxic-likeness >= 95th percentile',
                              'signal strength >= 98th percentile'],
           'caution_rules': ['ML uncertainty >= 15 points', 'gene list and ML differ by >= 40 points',
                             'chemistry >= 50% of SHAP contribution']}
json.dump(results, open(f'{BASE}/results/final_results.json', 'w'), indent=2)

n_best = sum(b.drug != 'none' for b in picks.best_per_hallmark)
print(f"False-alarm check: SHAP grouped by feature type (gene activity, chemistry, targets) for the top 20 candidates, "
      f"plus similarity to a toxic-drug signature and signature strength; {(r['red_flags'] != '').sum()} of {len(r)} "
      f"candidate pairs red-flagged. Final AI selection over {usable['drug'].nunique()} unflagged candidates: best drug "
      f"named for {n_best} of 6 hallmarks; top multi-hallmark candidates: {', '.join(p.drug for p in picks.top5) or 'none'}.")

False-alarm check: SHAP grouped by feature type (gene activity, chemistry, targets) for the top 20 candidates, plus similarity to a toxic-drug signature and signature strength; 76 of 90 candidate pairs red-flagged. Final AI selection over 13 unflagged candidates: best drug named for 6 of 6 hallmarks; top multi-hallmark candidates: kenpaullone, necrostatin-1, as-605240, ku-c103655, tgx-115.
